In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sys
import os

project_root = os.path.dirname(os.path.abspath(''))
if project_root not in sys.path:
    sys.path.append(project_root)

from FEATURES.features import *
from FEATURES.featuresV2 import *
from PRODUCTION.calculateEVS import *
# from PRODUCTION.pipelineV2 import *
from PRODUCTION.teamInfo import teamStarPlayer, projectedStartingFive, mainStartingFive

### Update projected starting lineups

In [ ]:
from MODELS.scrapStarting import NBADailyLineups

scraper = NBADailyLineups("https://www.rotowire.com/basketball/nba-lineups.php")
scraper.getDict()  # Scrape the lineups
scraper.updateTeamInfo()  # Update teamInfo.py

Successfully updated /Users/alexgonzalez/Documents/NBA-Prop-Predictor/PRODUCTION/teamInfo.py
Updated 16 teams with confirmed lineups


### Load Model

### Load Player Data and Bookmaker Data

In [2]:
pd.set_option('display.max_columns', None)
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

s26 = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_26.csv').sort_values(by='GAME_DATE')

usData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_US_{today}.csv')
dfsData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_DFS_{today}.csv')

dfsData.head()

,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE
0,Underdog,player_points,Myles Turner,Over,12.5,-137,2025-12-02,2025-12-01T18:42:34Z
1,Underdog,player_points,Myles Turner,Under,12.5,-137,2025-12-02,2025-12-01T18:42:34Z
2,Underdog,player_points,Khris Middleton,Over,11.5,-137,2025-12-02,2025-12-01T18:42:34Z
3,Underdog,player_points,Khris Middleton,Under,11.5,-137,2025-12-02,2025-12-01T18:42:34Z
4,Underdog,player_points,C.J. McCollum,Over,18.5,-137,2025-12-02,2025-12-01T18:42:34Z


In [3]:
from PRODUCTION.featureEngine.feature_engine import FeatureEngine

# Note: pts_model is optional - points prediction now uses Poisson model
engine = FeatureEngine({
    "min_model": "../MODELS/SAVED_MODELS/min_model.pkl",
    "usg_model": "../MODELS/SAVED_MODELS/usg_model.pkl",
})

result = engine.project_player(
    player_name="Lauri Markkanen",
    data=s26,
    date="2025-11-30",
    projectedStartingFive=projectedStartingFive,
    mainStartingFive=mainStartingFive,
    teamStarPlayer=teamStarPlayer,
    league_df=league_df,
    findOpp=findOpp
)

print(result)

{'predicted_minutes': 35.6701774597168, 'predicted_usage': 0.2721424400806427, 'predicted_points': 27.51808003013283}


## Top EVs for 2 leg bets

### Underdog picks

In [4]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

underdogPairs = calculate2LegBets(
    s26, dfsPTS, engine, current_date, 
    edge_threshold=4, stake=10, max_player_appearances=1, top_n=10,
    projectedStartingFive=projectedStartingFive,
    mainStartingFive=mainStartingFive,
    teamStarPlayer=teamStarPlayer,
    league_df=league_df,
    findOpp=findOpp
)

underdogPairs = underdogPairs[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2','ODDS 1', 'ODDS 2', 'PREDICTION 1', 'PREDICTION 2', 'PROB 1', 'PROB 2', 'MODEL SIDE 1', 'MODEL SIDE 2', 'RECOMMENDATION','EV%', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2']]
underdogPairs.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogPairs.csv', index=False)
underdogPairs.head()

Pre-computing predictions for 93 players...
Processing 84 players...
Generated 3321 valid 2-leg combinations


,NAME 1,NAME 2,LINE 1,LINE 2,ODDS 1,ODDS 2,PREDICTION 1,PREDICTION 2,PROB 1,PROB 2,MODEL SIDE 1,MODEL SIDE 2,RECOMMENDATION,EV%,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2
2321,Dru Smith,Jordan Goodwin,4.5,5.5,-125,-118,6.39,9.37,0.998,1.000,over,over,0,193.53,0.968,Low,Low
1740,Isaiah Jackson,Pelle Larsson,6.5,7.5,-137,-115,9.03,10.21,0.998,0.997,over,over,0,192.63,0.963,Low,Low
3221,Keyonte George,Rui Hachimura,19.5,11.5,-105,-117,25.37,14.86,0.994,0.991,over,over,0,189.76,0.949,Low,Low
3157,Spencer Jones,Svi Mykhailiuk,5.5,7.5,-137,-105,4.04,9.72,0.979,0.990,under,over,0,185.07,0.925,Low,Low
1137,Duncan Robinson,Brandon Miller,11.5,19.5,-108,-115,14.28,15.92,0.978,0.973,over,under,0,179.54,0.898,Low,Low


### Prizepicks picks

In [5]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points') & (dfsData['LINE'] < 24) & (dfsData['LINE'] > 5)]

prizepicksPairs = calculate2LegBets(
    s26, dfsPTS, engine, current_date, 
    edge_threshold=4, stake=10, max_player_appearances=1, top_n=10,
    projectedStartingFive=projectedStartingFive,
    mainStartingFive=mainStartingFive,
    teamStarPlayer=teamStarPlayer,
    league_df=league_df,
    findOpp=findOpp
)

pairsPrizepicks = prizepicksPairs[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2', 'PREDICTION 1', 'PREDICTION 2', 'PROB 1', 'PROB 2', 'MODEL SIDE 1', 'MODEL SIDE 2', 'RECOMMENDATION','EV%', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2']].head(10)
pairsPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksPairs.csv', index=False)
prizepicksPairs

Pre-computing predictions for 104 players...
Processing 92 players...
Generated 3971 valid 2-leg combinations


,NAME 1,NAME 2,LINE 1,LINE 2,ODDS 1,ODDS 2,PREDICTION 1,PREDICTION 2,MODEL SIDE 1,MODEL SIDE 2,PROB 1,PROB 2,PROB BOTH,EDGE 1,EDGE 2,COMBINED EDGE,EV%,KELLY FULL,RECOMMENDATION,SIGMA 1,SIGMA 2,SIGMA FLAG 1,SIGMA FLAG 2,CI 1,CI 2,CORRELATION,SAME_GAME,EXPECTED ROI
2036,Isaiah Jackson,Jordan Goodwin,6.5,5.5,-137,-118,9.03,9.37,over,over,0.998,1.000,0.9782,0.434,0.472,0.679,193.47,0.967,0,0.74,0.75,Low,Low,"(7.6, 10.5)","(7.9, 10.8)",0.05,0,193.5
2701,Pelle Larsson,Grayson Allen,7.5,13.5,-115,-118,10.21,18.27,over,over,0.997,0.998,0.9750,0.475,0.470,0.698,192.51,0.963,0,0.78,1.10,Low,Low,"(8.7, 11.7)","(16.1, 20.4)",0.05,0,192.5
3834,Keyonte George,Rui Hachimura,19.5,11.5,-105,-117,25.37,14.86,over,over,0.994,0.991,0.9659,0.494,0.465,0.701,189.76,0.949,0,1.24,0.95,Low,Low,"(23.0, 27.8)","(13.0, 16.7)",0.05,0,189.8
3774,Bruce Brown,Svi Mykhailiuk,8.0,7.5,-137,-105,6.29,9.72,under,over,0.983,0.990,0.9533,0.419,0.490,0.671,186.00,0.930,0,0.62,0.77,Low,Low,"(5.1, 7.5)","(8.2, 11.2)",0.05,0,186.0
1245,Duncan Robinson,Brandon Miller,11.5,19.5,-108,-115,14.28,15.92,over,under,0.978,0.973,0.9318,0.471,0.451,0.666,179.54,0.898,0,0.92,1.57,Low,Low,"(12.5, 16.1)","(12.8, 19.0)",0.05,0,179.5
1138,Tobias Harris,Austin Reaves,13.5,23.5,105,-105,16.42,28.63,over,over,0.967,0.971,0.9205,0.491,0.472,0.681,176.14,0.881,0,1.17,1.32,Low,Low,"(14.1, 18.7)","(26.0, 31.2)",0.05,0,176.1
2651,Davion Mitchell,Collin Gillespie,8.0,11.5,-137,-118,9.66,13.92,over,over,0.955,0.962,0.8997,0.391,0.433,0.600,169.91,0.850,0,0.76,0.91,Low,Low,"(8.2, 11.2)","(12.1, 15.7)",0.05,0,169.9
452,Kyle Kuzma,Kawhi Leonard,11.0,22.5,-137,-118,13.18,26.75,over,over,0.952,0.952,0.8881,0.388,0.424,0.589,166.44,0.832,0,0.89,1.57,Low,Low,"(11.4, 14.9)","(23.7, 29.8)",0.05,0,166.4
2221,Norman Powell,Miles Bridges,20.5,19.5,-120,-105,24.32,23.03,over,over,0.949,0.944,0.8779,0.417,0.444,0.611,163.37,0.817,0,1.22,1.17,Low,Low,"(21.9, 26.7)","(20.7, 25.3)",0.05,0,163.4
2538,Kel'el Ware,LeBron James,11.5,19.5,100,104,13.61,16.08,over,under,0.941,0.935,0.8623,0.453,0.457,0.628,158.68,0.793,0,0.90,1.93,Low,Low,"(11.8, 15.4)","(12.3, 19.9)",0.05,0,158.7


## 3 leg parlay

### Underdog picks

In [6]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points') & (dfsData['LINE'] < 24) & (dfsData['LINE'] > 5)]

underdogTrios = calculate3LegBets(
    s26, dfsPTS, engine, current_date, 
    edge_threshold=4, stake=10, max_player_appearances=1, top_n=10,
    projectedStartingFive=projectedStartingFive,
    mainStartingFive=mainStartingFive,
    teamStarPlayer=teamStarPlayer,
    league_df=league_df,
    findOpp=findOpp
)

underdogTrios = underdogTrios[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'PREDICTION 1', 'PREDICTION 2', 'PREDICTION 3', 'PROB 1', 'PROB 2', 'PROB 3', 'MODEL SIDE 1', 'MODEL SIDE 2', 'MODEL SIDE 3', 'RECOMMENDATION','EV%', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']].head(10)
underdogTrios.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogTrios.csv', index=False)
underdogTrios.head()

Pre-computing predictions for 81 players...
Processing 72 players...
Generated 51710 valid 3-leg combinations


,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,PROB 1,PROB 2,PROB 3,MODEL SIDE 1,MODEL SIDE 2,MODEL SIDE 3,RECOMMENDATION,EV%,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2,SIGMA FLAG 3
37226,Isaiah Jackson,Pelle Larsson,Jordan Goodwin,6.5,7.5,5.5,9.03,10.21,9.37,0.998,0.997,1.000,over,over,over,0,437.49,0.875,Low,Low,Low
51349,Spencer Jones,Keyonte George,Rui Hachimura,5.5,19.5,11.5,4.04,25.37,14.86,0.979,0.994,0.991,under,over,over,0,421.28,0.843,Low,Low,Low
26876,Duncan Robinson,Brandon Miller,Svi Mykhailiuk,11.5,19.5,7.5,14.28,15.92,9.72,0.978,0.973,0.990,over,under,over,0,408.27,0.817,Low,Low,Low
38091,Kawhi Leonard,Miles Bridges,Austin Reaves,22.5,19.5,23.5,26.75,23.03,28.63,0.952,0.944,0.971,over,over,over,0,371.29,0.743,Low,Low,Low
42370,Norman Powell,Anthony Davis,LeBron James,20.5,19.5,19.5,24.32,16.67,16.08,0.949,0.932,0.935,over,under,under,0,346.58,0.693,Low,Low,Low


### Prizepicks picks

In [7]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points') & (dfsData['LINE'] < 24) & (dfsData['LINE'] > 5)]

triosPrizepicks = calculate3LegBets(
    s26, dfsPTS, engine, current_date, 
    edge_threshold=4, stake=10, max_player_appearances=1, top_n=10,
    projectedStartingFive=projectedStartingFive,
    mainStartingFive=mainStartingFive,
    teamStarPlayer=teamStarPlayer,
    league_df=league_df,
    findOpp=findOpp
)

triosPrizepicks = triosPrizepicks[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'PREDICTION 1', 'PREDICTION 2', 'PREDICTION 3', 'PROB 1', 'PROB 2', 'PROB 3', 'MODEL SIDE 1', 'MODEL SIDE 2', 'MODEL SIDE 3', 'RECOMMENDATION','EV%', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']].head(10)
triosPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksTrios.csv', index=False)
triosPrizepicks.head()

Pre-computing predictions for 104 players...
Processing 92 players...
Generated 106804 valid 3-leg combinations


,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,PROB 1,PROB 2,PROB 3,MODEL SIDE 1,MODEL SIDE 2,MODEL SIDE 3,RECOMMENDATION,EV%,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2,SIGMA FLAG 3
70684,Isaiah Jackson,Pelle Larsson,Jordan Goodwin,6.5,7.5,5.5,9.03,10.21,9.37,0.998,0.997,1.000,over,over,over,0,437.49,0.875,Low,Low,Low
105947,Bruce Brown,Keyonte George,Grayson Allen,8.0,19.5,13.5,6.29,25.37,18.27,0.983,0.994,0.998,under,over,over,0,426.36,0.853,Low,Low,Low
48998,Duncan Robinson,Svi Mykhailiuk,Rui Hachimura,11.5,7.5,11.5,14.28,9.72,14.86,0.978,0.990,0.991,over,over,over,0,418.06,0.836,Low,Low,Low
42814,Tobias Harris,Brandon Miller,Austin Reaves,13.5,19.5,23.5,16.42,15.92,28.63,0.967,0.973,0.971,over,under,over,0,393.33,0.787,Low,Low,Low
19398,Kyle Kuzma,Davion Mitchell,Collin Gillespie,11.0,8.0,11.5,13.18,9.66,13.92,0.952,0.955,0.962,over,over,over,0,372.01,0.744,Low,Low,Low
